In [ ]:
import torch 

print(torch.__version__)

: 

In [6]:
import torch 

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_name(1))

True
NVIDIA GeForce RTX 4090
NVIDIA GeForce RTX 4090


In [8]:
import torch
import time

# ------------------------------------------------------------
# 목적:
#   "같은 계산을 CPU로 할 때" vs "GPU로 할 때" 시간이 얼마나 차이나는지 체험한다.
#
# 핵심 아이디어:
#   - 딥러닝/머신러닝에서 가장 많이 하는 연산이 '큰 행렬 곱셈'이다.
#   - CPU는 소수의 코어로 순차/병렬을 하고,
#   - GPU는 매우 많은 코어로 동일한 형태의 계산을 대량 병렬 처리한다.
#   => 그래서 큰 행렬 연산에서는 GPU가 압도적으로 빠른 경우가 많다.
# ------------------------------------------------------------

# 행렬의 크기 설정
# N이 너무 작으면 CPU도 금방 끝나서 GPU와 차이가 잘 안 보일 수 있다.
# 너무 크면 메모리 부족(OOM)이 날 수 있다.
# 보통 2000~6000 사이에서 환경에 맞춰 조절하면 된다.
N = 4000

# ------------------------------------------------------------
# 1) CPU에서 행렬 곱셈 시간 측정
# ------------------------------------------------------------

# torch.randn(N, N):
#   평균 0, 분산 1인 정규분포에서 난수를 뽑아 (N x N) 행렬(텐서)을 만든다.
#   여기서는 "큰 계산"을 만들어내기 위한 재료일 뿐, 의미 있는 데이터는 아니다.
a_cpu = torch.randn(N, N)
b_cpu = torch.randn(N, N)

# time.time():
#   현재 시간을 초 단위로 반환한다.
#   시작 시각을 저장해두고, 연산이 끝난 시각과 빼면 걸린 시간이 된다.
start = time.time()

# @ 연산자:
#   PyTorch에서 텐서의 행렬 곱(matrix multiplication)을 의미한다.
#   (N x N) @ (N x N) -> (N x N)
#   이 계산이 바로 "딥러닝에서 반복되는 핵심 연산"의 대표 예시다.
c_cpu = a_cpu @ b_cpu

# CPU에서 걸린 시간(초)
cpu_time = time.time() - start
print(f"CPU time: {cpu_time:.2f} sec")

# ------------------------------------------------------------
# 2) GPU가 있는 경우, GPU에서 같은 연산 시간 측정
# ------------------------------------------------------------

# torch.cuda.is_available():
#   현재 환경에서 CUDA GPU를 사용할 수 있는지(True/False) 알려준다.
#   - True: NVIDIA GPU가 있고, 드라이버/쿠다가 정상이며, PyTorch가 CUDA 지원 빌드임
#   - False: GPU가 없거나(혹은) 드라이버/환경이 맞지 않는 경우
if torch.cuda.is_available():
    # device = "cuda" 는 GPU를 의미한다.
    # (CPU는 "cpu")
    device = torch.device("cuda")

    # --------------------------------------------------------
    # 2-1) CPU에 있는 데이터를 GPU로 옮기기
    # --------------------------------------------------------
    # to(device):
    #   텐서를 지정한 장치로 이동시킨다.
    #   여기서는 CPU 텐서를 GPU VRAM으로 복사한다.
    #   (이 복사 자체도 시간이 걸리므로, "연산 속도 비교"에서는 보통 연산만 재려면
    #    복사 시간은 제외하고 재는 편이다.)
    a_gpu = a_cpu.to(device)
    b_gpu = b_cpu.to(device)

    # --------------------------------------------------------
    # 2-2) GPU 연산 시간 측정 시 주의사항: GPU는 비동기(async)로 동작한다
    # --------------------------------------------------------
    # GPU 연산은 "시켜놓고" CPU가 바로 다음 줄로 넘어갈 수 있다.
    # 그래서 time.time()으로 바로 재면 실제 연산이 끝나기 전에 시간이 찍힐 수 있다.
    #
    # torch.cuda.synchronize():
    #   GPU가 이전에 받은 작업을 "모두 끝낼 때까지" 기다린다.
    #   정확한 시간 측정을 위해 '시작 전'과 '끝난 후'에 synchronize()를 호출한다.

    torch.cuda.synchronize()  # 측정 시작 전에 GPU 작업이 남아있지 않도록 정리
    start = time.time()

    # GPU에서 행렬 곱셈 수행
    # (텐서가 이미 GPU에 있으므로 GPU에서 계산된다.)
    c_gpu = a_gpu @ b_gpu

    torch.cuda.synchronize()  # 연산이 실제로 끝날 때까지 기다린 후 시간 측정
    gpu_time = time.time() - start

    print(f"GPU time: {gpu_time:.2f} sec")

    # --------------------------------------------------------
    # 2-3) 추가로 보여주면 좋은 정보(수업용)
    # --------------------------------------------------------
    # GPU 이름 출력 (학생들이 "아 이 서버에는 이런 GPU가 있구나" 체감)
    print("GPU name:", torch.cuda.get_device_name(0))

else:
    # GPU가 없으면 여기로 온다.
    print("GPU not available")

CPU time: 0.11 sec
GPU time: 0.00 sec
GPU name: NVIDIA GeForce RTX 4090
